# Module 6.4 — Multi-Query Retriever

A single query may miss relevant documents due to phrasing. MultiQueryRetriever generates **multiple rephrased queries** using an LLM and **unions** the results → better recall.

```
Original query
      ↓ LLM generates
  [Query variant 1] → retrieved docs
  [Query variant 2] → retrieved docs    } de-duplicated union
  [Query variant 3] → retrieved docs
```

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.schema import Document
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

docs = [
    Document(page_content="Climate change refers to long-term shifts in temperatures and weather patterns."),
    Document(page_content="Global warming is primarily caused by human activities since the 1800s."),
    Document(page_content="Greenhouse gases trap heat in the atmosphere, raising Earth's temperature."),
    Document(page_content="Renewable energy like solar and wind can help reduce carbon emissions."),
    Document(page_content="The Paris Agreement aims to limit global warming to 1.5°C above pre-industrial levels."),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="mq_demo")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=vs.as_retriever(search_kwargs={"k": 3}),
    llm=llm
)

query   = "What causes environmental temperature changes?"
results = multi_retriever.invoke(query)
print(f"\nQuery: '{query}'")
print(f"Unique docs retrieved: {len(results)}")
for d in results:
    print(f"  • {d.page_content}")
